In [1]:
from content.connection.credentials import read_credentials
from content.connection.connect_db import create_conection
from content.extraction.read_db import read_file

import pandas as pd

In [2]:

credentials = read_credentials()
engine_local = create_conection(credentials)

Conexion exitosa a la DB


In [3]:
df_mail_repository, df_sinfin_repository = read_file(engine_local)

In [4]:
df_mail_repository = df_mail_repository.sort_values(by=['CTA CONTR', 'FECHA DE CESION'], ascending=[True, True])
df_mail_repository = df_mail_repository.drop_duplicates(subset=['CTA CONTR'])
df_sinfin_repository = df_sinfin_repository.sort_values(by=['NUMERO_CUENTA', 'DATE1'], ascending=[True, True])
df_sinfin_repository = df_sinfin_repository.drop_duplicates(subset=['NUMERO_CUENTA'])
df_mail_repository = df_mail_repository[['CTA CONTR', 'CORREO ELECTRONICO']]
df_repository = pd.merge(df_mail_repository, df_sinfin_repository, how='outer', left_on='CTA CONTR', right_on='NUMERO_CUENTA')
df_repository['EmailsDeudor'] = df_repository['CORREO ELECTRONICO']
df_repository = df_repository.drop(columns=['CTA CONTR', 'CORREO ELECTRONICO'])

#df_repository.to_sql('asignacion', engine_local, if_exists='append', index=False, schema='sinfin')

In [5]:
from content.db.querys import read_asignation_naturgy

text_query = read_asignation_naturgy()

df_asignation_db = pd.read_sql_query(text_query, engine_local)


In [6]:
df_new_account = pd.merge(df_asignation_db, df_repository, how='right', left_on='NUMERO_CUENTA', right_on='NUMERO_CUENTA')
print(len(df_new_account))

19784


In [7]:
df_new = pd.merge(df_asignation_db, df_repository, how='inner', left_on='NUMERO_CUENTA', right_on='NUMERO_CUENTA', suffixes=('_x',''))
columns_drop = [col for col in df_new.columns if col.endswith('_x')]
df_new = df_new.drop(columns=columns_drop)
df_new = df_new.reset_index()
print(len(df_new))

6773


In [8]:
df_new['NUMERO_CUENTA'] = pd.to_numeric(df_new['NUMERO_CUENTA'], errors='coerce')

In [9]:
from sqlalchemy import text


with engine_local.connect() as connection:
    for index, row in df_new.iterrows():
        update_query = text("""
                            UPDATE      sinfin.asignacion
                            SET         "MONEY1" = :MONEY1,
                                        "MONEY4" = :MONEY4,
                                        "DATE1" = :DATE1,
                                        "DATE2" = :DATE2,
                                        "TEXT1" = :TEXT1,
                                        "TEXT2" = :TEXT2,
                                        "TEXT3" = :TEXT3,
                                        "TEXT4" = :TEXT4,
                                        "TEXT5" = :TEXT5,
                                        "TEXT6" = :TEXT6,
                                        "TEXT7" = :TEXT7,
                                        "TEXT8" = :TEXT8,
                                        "TEXT9" = :TEXT9
                            WHERE       "NUMERO_CUENTA" = :NUMERO_CUENTA   
                            """)
        params=  { 
                            'MONEY1' : row['MONEY1'],
                            'MONEY4' : row['MONEY4'],
                            'DATE1' : row['DATE1'],
                            'DATE2' : row['DATE2'],
                            'TEXT1' : row['TEXT1'],
                            'TEXT2' : row['TEXT2'],
                            'TEXT3' : row['TEXT3'],
                            'TEXT4' : row['TEXT4'],
                            'TEXT5' : row['TEXT5'],
                            'TEXT6' : row['TEXT6'],
                            'TEXT7' : row['TEXT7'],
                            'TEXT8' : row['TEXT8'],
                            'TEXT9' : row['TEXT9'],
                            'NUMERO_CUENTA' : row['NUMERO_CUENTA']
                            }
        print(f'Ejecución query con los parametros {params}')
        result = connection.execute(update_query, params)
        print(f'Filas afectadas: {result.rowcount}')

Ejecución query con los parametros {'MONEY1': 131.05, 'MONEY4': 131.05, 'DATE1': Timestamp('2024-07-18 00:00:00'), 'DATE2': Timestamp('2024-10-23 00:00:00'), 'TEXT1': 132.0, 'TEXT2': 'BAJA - VENCIDA', 'TEXT3': nan, 'TEXT4': 9754715847625.0, 'TEXT5': '9050708431090003975471584762518072400000131050', 'TEXT6': 246100047786, 'TEXT7': 'E0022', 'TEXT8': '3 VTA', 'TEXT9': 14805726.0, 'NUMERO_CUENTA': 47158476}
Filas afectadas: 1
Ejecución query con los parametros {'MONEY1': 130.87, 'MONEY4': 130.87, 'DATE1': Timestamp('2024-05-27 00:00:00'), 'DATE2': Timestamp('2024-07-08 00:00:00'), 'TEXT1': 132.0, 'TEXT2': 'ACTIVA - VIGENTE', 'TEXT3': 'G0230', 'TEXT4': 9754715897552.0, 'TEXT5': '9050708431090003975471589755227052400000130870', 'TEXT6': 242100047544, 'TEXT7': 'G0230', 'TEXT8': 'NR21', 'TEXT9': 14533591.0, 'NUMERO_CUENTA': 47158975}
Filas afectadas: 1
Ejecución query con los parametros {'MONEY1': 800.69, 'MONEY4': 800.69, 'DATE1': Timestamp('2024-06-03 00:00:00'), 'DATE2': Timestamp('2024-07-

KeyboardInterrupt: 

In [10]:
from psycopg2.extras import execute_batch

cur = engine_local.cursor()
# Ejecutar las actualizaciones en bloque
execute_batch(cur, """
    UPDATE tabla_principal
    SET "MONEY1" = %s,
        "MONEY4" = %s
    WHERE id = %s
""", df_new)

engine_local.commit()
cur.close()
engine_local.close()

Filas afectadas: 1
